# 🔧 Fine-tune Reranker — END-TO-END (Kaggle 2×T4)

Phá trần off-the-shelf **0.5766** → mục tiêu ~0.65+. Một mạch: **synthetic → hard-neg → train → eval (GATE) → retrieval với FT reranker**.

**Settings (phải):** Accelerator = **GPU T4 ×2** · Internet = **On**.
**Add Input:** `corpus_articles.jsonl` (hf 93K) + `corpus_emb.npy` (của nó) + `stage1_questions.json`.

**Thời gian:** F1 gen 2-4h · F2 mine ~0.5h · F3 train 1.5-3h · F4 eval ~0.2h · Phase A ~1h. Có thể tách phiên (Save Version giữ output).
⚠️ Script đã review đối kháng nhưng **chưa chạy thực Kaggle** — lần đầu có thể vướng runtime (OOM → giảm `train_group_size` 8→6; entrypoint → đã pin `FlagEmbedding>=1.3`). Báo log nếu lỗi.

## 0. Setup — clone repo + cài FlagEmbedding

In [ ]:
import os, subprocess
if os.path.isdir("/kaggle/working/ai_legal_assistant"):
    get_ipython().system('cd /kaggle/working/ai_legal_assistant && git pull -q')
else:
    get_ipython().system('cd /kaggle/working && git clone -q https://github.com/lehuubao2909/ai_legal_assistant.git')
os.chdir("/kaggle/working/ai_legal_assistant")
get_ipython().system('pip install -q -U "FlagEmbedding>=1.3" deepspeed')
import importlib; importlib.import_module("FlagEmbedding")   # assert cài được trước khi train
get_ipython().system('nvidia-smi -L')
print("repo:", os.getcwd())

## 1. Phase F1 — Synthetic queries (Qwen2.5-7B, ~2-4h)
Sinh câu hỏi aspect-guided từ mỗi điều → lọc BGE-M3 top-40. Ghi `/kaggle/working/ft/synth_pairs.jsonl` (resumable). **Save Version sau bước này** để giữ synth_pairs.

In [ ]:
get_ipython().system('python scratch/finetune/gen_synthetic_pairs.py --max-articles 20000')

## 2. Phase F2 — Hard negatives (BGE-M3, ~0.5h)
Mine hard-neg ở rank [10,60) (tránh false-neg), tái dùng `corpus_emb.npy`. Ghi `train_reranker.jsonl` (FlagEmbedding format) + `eval_pairs.jsonl` (held-out 10%).

In [ ]:
get_ipython().system('python scratch/finetune/mine_hard_negatives.py')

## 3. Phase F3 — Train reranker (FlagEmbedding DDP 2×T4, ~1.5-3h)
Fine-tune `AITeamVN/Vietnamese_Reranker` → `/kaggle/working/ft_reranker`. OOM? → `--train-group-size 6`.

In [ ]:
get_ipython().system('python scratch/finetune/train_reranker.py')

## 4. Phase F4 — ⛔ GATE: eval base vs fine-tuned (offline)
MRR@10 + recall@{1,3,5,10}. **CHỈ sang Phase A + nộp nếu FT > base rõ rệt.** Không thì sửa (xem plan) rồi train lại — đừng phí lượt leaderboard.

In [ ]:
get_ipython().system('python scratch/finetune/eval_reranker.py')

## 5. Phase A — Retrieval với reranker FINE-TUNE → `retrieved.json`
Tái dùng đúng pipeline đã ra 0.5766, chỉ đổi reranker sang FT (RERANK_MAX=512 khớp training). Xóa `retrieved.json` cũ trong working trước khi chạy lại.

In [ ]:
# tìm corpus + questions (như full_pipeline) — định nghĩa WORK/CORPUS_PATH/questions cho cell retrieval
import os, glob, json
WORK = "/kaggle/working"
def find_file(name):
    for p in [os.path.join(WORK, name), name] + glob.glob(f"/kaggle/input/**/{name}", recursive=True):
        if os.path.exists(p): return p
    raise FileNotFoundError(f"Không thấy {name} — Add Input.")
CORPUS_PATH = find_file("corpus_articles.jsonl")
questions = json.load(open(find_file("stage1_questions.json"), encoding="utf-8"))
print("corpus:", CORPUS_PATH, "| questions:", len(questions))

In [ ]:
import os, glob
import re, numpy as np, torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rank_bm25 import BM25Okapi

EMBED_ID = "AITeamVN/Vietnamese_Embedding"
FT_DIR = "/kaggle/working/ft_reranker"
assert os.path.isdir(FT_DIR), "Chưa có ft_reranker — chạy Phase F3 (train) trước!"
RERANK_ID = FT_DIR   # <<< RERANKER ĐÃ FINE-TUNE (đổi về AITeamVN/Vietnamese_Reranker nếu muốn so base)
# Retrieval với reranker FINE-TUNE. RERANK_MAX=512 KHỚP training (query_max 64 + passage_max 448)
# → không train/serve skew. corpus_emb hf 93K TÁI DÙNG (corpus không đổi). Xóa retrieved.json để chạy lại.
MAX_SEQ, RERANK_MAX, CAND, CAND_SAVE, RR_BATCH = 1024, 512, 80, 20, 16
RRF_K, W_BM25, W_DENSE = 60, 0.65, 0.35   # RRF có trọng số — ưu BM25 (thuật ngữ luật). Đặt 0.5/0.5 = cũ.
DEV = "cuda" if torch.cuda.is_available() else "cpu"
RET_PATH = f"{WORK}/retrieved.json"
EMB_PATH = f"{WORK}/corpus_emb.npy"
def tok(t): return re.findall(r"\w+", (t or "").lower(), flags=re.UNICODE)

if os.path.exists(RET_PATH):
    cache = {int(k): v for k, v in json.load(open(RET_PATH, encoding="utf-8")).items()}
    print(f"Đã có retrieved.json ({len(cache)} câu) → BỎ QUA Phase A. (Muốn chạy lại CAND=50: xóa file này)")
else:
    corpus = [json.loads(l) for l in open(CORPUS_PATH, encoding="utf-8")
              if l.strip() and json.loads(l).get("doc_number")]
    docs_text = [f"{r['title']}\n{r['text']}" for r in corpus]
    print("corpus:", len(corpus), "| device:", DEV, "| phễu rerank CAND:", CAND)

    emb = SentenceTransformer(EMBED_ID, device=DEV); emb.max_seq_length = MAX_SEQ
    emb_in = EMB_PATH if os.path.exists(EMB_PATH) else (glob.glob("/kaggle/input/**/corpus_emb.npy", recursive=True) or [None])[0]
    corpus_emb = None
    if emb_in and os.path.exists(emb_in):
        _e = np.load(emb_in).astype("float32")
        if len(_e) == len(corpus):
            corpus_emb = _e; print("Dùng corpus_emb.npy có sẵn (khớp", len(corpus), "dòng) → BỎ QUA embed")
        else:
            print(f"⚠ corpus_emb.npy ({len(_e)}) ≠ corpus ({len(corpus)}) → BỎ emb cũ, embed lại corpus mới")
    if corpus_emb is None:   # corpus vbpl MỚI (40939) ≠ HF cũ (93K) → embed lại (~45p T4, nhanh hơn 93K)
        corpus_emb = emb.encode(docs_text, batch_size=128, normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True).astype("float32")
        np.save(EMB_PATH, corpus_emb); print("Đã lưu", EMB_PATH, f"({len(corpus_emb)} vec)")
    q_emb = emb.encode([q["question"] for q in questions], batch_size=128, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype("float32")
    bm25 = BM25Okapi([tok(t) for t in docs_text])
    rr_tok = AutoTokenizer.from_pretrained(RERANK_ID)
    rr = AutoModelForSequenceClassification.from_pretrained(RERANK_ID).to(DEV).eval()
    if DEV == "cuda": rr = rr.half()

    @torch.no_grad()
    def rerank(query, idxs):
        pairs = [[query, docs_text[i]] for i in idxs]; out = []
        for j in range(0, len(pairs), RR_BATCH):
            enc = rr_tok(pairs[j:j+RR_BATCH], padding=True, truncation=True,
                         max_length=RERANK_MAX, return_tensors="pt").to(DEV)
            out.extend(rr(**enc).logits.view(-1).float().cpu().tolist())
        return out

    cache = {}
    for n, q in enumerate(questions):
        dense = np.argsort(corpus_emb @ q_emb[n])[::-1][:CAND]
        bm = bm25.get_scores(tok(q["question"])); sp = [i for i in np.argsort(bm)[::-1][:CAND] if bm[i] > 0]
        rrf = {}   # RRF có trọng số: ưu BM25 (số luật/điều/thuật ngữ khớp chính xác)
        for rank, i in enumerate(dense): rrf[int(i)] = rrf.get(int(i), 0) + W_DENSE/(RRF_K+rank+1)
        for rank, i in enumerate(sp):    rrf[int(i)] = rrf.get(int(i), 0) + W_BM25/(RRF_K+rank+1)
        fused = sorted(rrf, key=rrf.get, reverse=True)[:CAND]
        scored = sorted(zip(fused, rerank(q["question"], fused)), key=lambda x: x[1], reverse=True)
        seen, dd = set(), []
        for i, s in scored:
            k = (corpus[i]["doc_number"], corpus[i]["article"])
            if k not in seen: seen.add(k); dd.append((i, s))
        # Lưu top-CAND_SAVE candidate KÈM điểm rerank → cutoff áp ở Phase B (sweep offline được)
        cache[int(q["id"])] = [
            {**{k: corpus[i][k] for k in ("doc_number","clean_name","article","title","text")}, "score": float(s)}
            for i, s in dd[:CAND_SAVE]
        ]
        if (n+1) % 200 == 0: print(f"retrieved {n+1}/{len(questions)}")

    json.dump({str(k): v for k, v in cache.items()}, open(RET_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    print("Đã lưu", RET_PATH, f"(top-{CAND_SAVE} + score, phễu rerank {CAND}) → giải phóng VRAM Phase A")
    del emb, rr, corpus_emb, q_emb; import gc; gc.collect(); torch.cuda.empty_cache()

## 6. Tải `retrieved.json` (tab Output) → sweep local
```bash
python scratch/sweep_cutoff.py --retrieved backup/time8/retrieved.json --base /tmp/none.json
```
Nộp `submission_t5m3.zip` trước (điểm vận hành baseline) → so **0.5766**. FT thắng rõ → giữ; cập nhật `RERANK_ID` trong `full_pipeline_kaggle.ipynb` cell 10 = `/kaggle/working/ft_reranker` (hoặc Add-Input dataset) cho các lần sau.